# Explainability: Grad-CAM (OPTIONAL)

---

> **Note:** This notebook is **optional**. It covers advanced explainability techniques that are not required for the core curriculum but are highly valuable for real-world applications.

## Learning Objectives

By the end of this notebook, you will:

- Understand **why explainability matters** in computer vision
- Implement **Grad-CAM** (Gradient-weighted Class Activation Mapping) from scratch
- Produce and interpret **heatmaps** showing which image regions the model focuses on
- Overlay heatmaps on input images for visual explanation
- Know other explainability methods: saliency maps, LIME, SHAP

## Prerequisites

- **DL300 Notebooks 01-02**: CNN basics and training
- Understanding of convolutional layers, feature maps, and gradients

## Table of Contents

1. [Setup and Imports](#1.-Setup-and-Imports)
2. [Why Explainability Matters](#2.-Why-Explainability-Matters)
3. [Grad-CAM: Theory](#3.-Grad-CAM:-Theory)
4. [Data and Model Preparation](#4.-Data-and-Model-Preparation)
5. [Implementing Grad-CAM from Scratch](#5.-Implementing-Grad-CAM-from-Scratch)
6. [Visualizing Grad-CAM Heatmaps](#6.-Visualizing-Grad-CAM-Heatmaps)
7. [Applying Grad-CAM to Multiple Images](#7.-Applying-Grad-CAM-to-Multiple-Images)
8. [Other Explainability Methods](#8.-Other-Explainability-Methods)
9. [Common Mistakes and Debugging Tips](#9.-Common-Mistakes-and-Debugging-Tips)
10. [Exercises](#10.-Exercises)

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Guard optional dependencies
try:
    from torchvision import datasets, transforms, models
    HAS_TORCHVISION = True
except ImportError:
    HAS_TORCHVISION = False
    print("WARNING: torchvision not available. Some features will be limited.")

try:
    import cv2
    HAS_CV2 = True
except ImportError:
    HAS_CV2 = False
    print("Note: OpenCV (cv2) not available. Using matplotlib for heatmap overlay.")

import os

set_global_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print(f"torchvision available: {HAS_TORCHVISION}")
print(f"OpenCV available: {HAS_CV2}")
print("Setup complete.")

---

## 2. Why Explainability Matters

Deep neural networks are often called "black boxes" -- they achieve high accuracy, but it is hard to understand *why* they make specific predictions.

### Reasons to care about explainability

- **Trust**: Before deploying a model in healthcare, autonomous driving, or finance, stakeholders need to understand its reasoning
- **Debugging**: If a model performs poorly, explainability helps identify *what* it is looking at (maybe the wrong features)
- **Bias detection**: A model might achieve high accuracy by relying on spurious correlations (e.g., background instead of object)
- **Regulatory compliance**: Some industries require model explanations (e.g., GDPR "right to explanation")
- **Scientific insight**: Understanding what features matter can reveal new domain knowledge

### The Problem

A CNN with millions of parameters transforms an input image through dozens of layers. The question is:

$$\text{Which regions of the input image } \mathbf{x} \text{ most influenced the prediction } \hat{y}\text{?}$$

---

## 3. Grad-CAM: Theory

**Grad-CAM** (Gradient-weighted Class Activation Mapping) by Selvaraju et al. (2017) uses the gradients flowing into the **final convolutional layer** to produce a heatmap highlighting important regions.

### Algorithm

Given:
- A CNN with final conv layer producing feature maps $A^k \in \mathbb{R}^{H \times W}$ for channel $k$
- A target class $c$ with score (logit) $y^c$

**Step 1**: Compute the importance weights by global-average-pooling the gradients:

$$\alpha_k^c = \frac{1}{Z} \sum_i \sum_j \frac{\partial y^c}{\partial A^k_{ij}}$$

where $Z = H \times W$ (spatial dimensions of the feature map).

**Step 2**: Compute the weighted combination of feature maps, followed by ReLU:

$$L_{\text{Grad-CAM}}^c = \text{ReLU}\left(\sum_k \alpha_k^c \cdot A^k\right)$$

- The ReLU keeps only **positive** contributions (regions that increase the score for class $c$)
- The result is a coarse heatmap of size $(H, W)$, which is upsampled to the input image size

### Intuition

- $\alpha_k^c$ tells us how important feature map $k$ is for class $c$
- Combining all feature maps with these weights gives us a spatial map of importance
- Regions with high values = regions the model "looks at" for class $c$

---

## 4. Data and Model Preparation

In [ ]:
# Build a simple CNN and train on MNIST/CIFAR-10
data_dir = os.path.join("..", "..", "data")

# We use MNIST for simplicity (works without internet if already downloaded)
if HAS_TORCHVISION:
    try:
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),
        ])
        train_dataset = datasets.MNIST(data_dir, train=True, download=True, transform=transform)
        test_dataset = datasets.MNIST(data_dir, train=False, download=True, transform=transform)
        DATASET_NAME = "MNIST"
        N_CHANNELS = 1
        N_CLASSES = 10
        class_names = [str(i) for i in range(10)]
        MEAN, STD = (0.1307,), (0.3081,)
        print(f"Loaded {DATASET_NAME}: {len(train_dataset)} train, {len(test_dataset)} test")
    except Exception as e:
        print(f"Dataset loading failed: {e}")
        print("Will use synthetic data.")
        HAS_TORCHVISION = False

if not HAS_TORCHVISION:
    # Fallback: create synthetic data
    print("Using synthetic data for demonstration.")
    N_CHANNELS = 1
    N_CLASSES = 10
    class_names = [str(i) for i in range(10)]
    MEAN, STD = (0.5,), (0.5,)

In [ ]:
class SimpleCNN(nn.Module):
    """Simple CNN suitable for Grad-CAM demonstration.

    We keep the architecture simple and expose the last conv layer
    for Grad-CAM hook registration.
    """

    def __init__(self, n_channels=1, n_classes=10):
        super().__init__()

        # Convolutional feature extractor
        self.conv1 = nn.Conv2d(n_channels, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # <-- target for Grad-CAM
        self.bn3 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2)

        # Classifier
        # After 3 pools on 28x28: 28 -> 14 -> 7 -> 3
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # last conv output
        x = self.classifier(x)
        return x


# Train the model
set_global_seed(42)
model = SimpleCNN(n_channels=N_CHANNELS, n_classes=N_CLASSES).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")
print(model)

In [ ]:
# Quick training
if HAS_TORCHVISION:
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=128, num_workers=0)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    EPOCHS = 5
    for epoch in range(1, EPOCHS + 1):
        model.train()
        correct, total = 0, 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            logits = model(X_b)
            loss = loss_fn(logits, y_b)
            loss.backward()
            optimizer.step()
            correct += (logits.argmax(-1) == y_b).sum().item()
            total += y_b.size(0)

        train_acc = correct / total

        # Test accuracy
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for X_b, y_b in test_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                correct += (model(X_b).argmax(-1) == y_b).sum().item()
                total += y_b.size(0)
        test_acc = correct / total

        print(f"Epoch {epoch}/{EPOCHS} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")

    print(f"\nFinal test accuracy: {test_acc:.4f}")
else:
    print("Skipping training (no dataset available).")

---

## 5. Implementing Grad-CAM from Scratch

We implement Grad-CAM using PyTorch **hooks** to capture:
1. The **activations** (forward hook) of the target conv layer
2. The **gradients** (backward hook) flowing into that layer

In [ ]:
class GradCAM:
    """Grad-CAM implementation using PyTorch hooks.

    Args:
        model: Trained CNN model.
        target_layer: The convolutional layer to compute Grad-CAM for.
    """

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer

        # Storage for activations and gradients
        self.activations = None
        self.gradients = None

        # Register hooks
        self._forward_hook = target_layer.register_forward_hook(self._save_activation)
        self._backward_hook = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        """Forward hook: save the output activations."""
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        """Backward hook: save the gradients."""
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class=None):
        """Generate Grad-CAM heatmap.

        Args:
            input_tensor: Input image tensor of shape (1, C, H, W).
            target_class: Class index to generate heatmap for.
                          If None, uses the predicted class.

        Returns:
            heatmap: Numpy array of shape (H, W) with values in [0, 1].
        """
        self.model.eval()

        # Forward pass
        output = self.model(input_tensor)

        if target_class is None:
            target_class = output.argmax(dim=1).item()

        # Zero all gradients
        self.model.zero_grad()

        # Backward pass for the target class
        target_score = output[0, target_class]
        target_score.backward()

        # Step 1: Compute importance weights (global average pooling of gradients)
        # gradients shape: (1, C, H, W)
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)

        # Step 2: Weighted combination of feature maps + ReLU
        # activations shape: (1, C, H, W)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # (1, 1, H, W)
        cam = F.relu(cam)  # Keep only positive contributions

        # Step 3: Normalize to [0, 1]
        cam = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = cam / cam.max()

        return cam, target_class, output

    def remove_hooks(self):
        """Remove the registered hooks."""
        self._forward_hook.remove()
        self._backward_hook.remove()


# Create GradCAM object targeting the last conv layer
grad_cam = GradCAM(model, model.conv3)
print("Grad-CAM initialized.")
print(f"Target layer: conv3 ({model.conv3})")

---

## 6. Visualizing Grad-CAM Heatmaps

In [ ]:
def denormalize(tensor, mean, std):
    """Reverse normalization for display."""
    mean = torch.tensor(mean).view(-1, 1, 1)
    std = torch.tensor(std).view(-1, 1, 1)
    return (tensor * std + mean).clamp(0, 1)


def overlay_heatmap(image_np, heatmap, alpha=0.4):
    """Overlay heatmap on image.

    Args:
        image_np: Grayscale or RGB image as numpy array, shape (H, W) or (H, W, 3).
        heatmap: Heatmap array, shape (h, w), values in [0, 1].
        alpha: Transparency of the heatmap overlay.

    Returns:
        Blended image as numpy array.
    """
    from matplotlib.colors import Normalize
    from matplotlib.cm import ScalarMappable

    # Resize heatmap to match image dimensions
    if image_np.ndim == 2:
        h, w = image_np.shape
    else:
        h, w = image_np.shape[:2]

    # Upsample heatmap using bilinear interpolation
    heatmap_tensor = torch.tensor(heatmap, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    heatmap_resized = F.interpolate(heatmap_tensor, size=(h, w), mode="bilinear", align_corners=False)
    heatmap_resized = heatmap_resized.squeeze().numpy()

    # Apply colormap
    sm = ScalarMappable(norm=Normalize(0, 1), cmap="jet")
    heatmap_colored = sm.to_rgba(heatmap_resized)[:, :, :3]  # (H, W, 3)

    # Ensure image is RGB
    if image_np.ndim == 2:
        image_rgb = np.stack([image_np] * 3, axis=-1)
    else:
        image_rgb = image_np.copy()

    # Blend
    blended = (1 - alpha) * image_rgb + alpha * heatmap_colored
    return np.clip(blended, 0, 1)


print("Visualization utilities defined.")

In [ ]:
# Generate Grad-CAM for a single image
if HAS_TORCHVISION:
    # Pick a test image
    test_img, test_label = test_dataset[0]
    input_tensor = test_img.unsqueeze(0).to(device)  # (1, C, H, W)

    # Generate heatmap
    heatmap, predicted_class, logits = grad_cam.generate(input_tensor)

    # Denormalize image for display
    img_display = denormalize(test_img, MEAN, STD).squeeze().numpy()

    # Create overlay
    overlay = overlay_heatmap(img_display, heatmap, alpha=0.5)

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    # Original image
    axes[0].imshow(img_display, cmap="gray")
    axes[0].set_title(f"Original (Label: {class_names[test_label]})")
    axes[0].axis("off")

    # Heatmap
    heatmap_upsampled = F.interpolate(
        torch.tensor(heatmap).unsqueeze(0).unsqueeze(0).float(),
        size=img_display.shape[:2], mode="bilinear", align_corners=False
    ).squeeze().numpy()
    axes[1].imshow(heatmap_upsampled, cmap="jet")
    axes[1].set_title(f"Grad-CAM (Pred: {class_names[predicted_class]})")
    axes[1].axis("off")

    # Overlay
    axes[2].imshow(overlay)
    axes[2].set_title("Overlay")
    axes[2].axis("off")

    probs = F.softmax(logits, dim=1)
    confidence = probs[0, predicted_class].item()
    plt.suptitle(f"Grad-CAM Visualization (Confidence: {confidence:.2%})", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping (no dataset available).")

---

## 7. Applying Grad-CAM to Multiple Images

In [ ]:
if HAS_TORCHVISION:
    # Select diverse examples (one per class)
    examples_per_class = {}
    for idx in range(len(test_dataset)):
        _, label = test_dataset[idx]
        if label not in examples_per_class:
            examples_per_class[label] = idx
        if len(examples_per_class) == N_CLASSES:
            break

    # Generate Grad-CAM for each class
    fig, axes = plt.subplots(3, N_CLASSES, figsize=(20, 6))

    for col, (label, idx) in enumerate(sorted(examples_per_class.items())):
        test_img, test_label = test_dataset[idx]
        input_tensor = test_img.unsqueeze(0).to(device)

        # Generate heatmap
        heatmap, pred_class, logits = grad_cam.generate(input_tensor)

        # Denormalize
        img_display = denormalize(test_img, MEAN, STD).squeeze().numpy()

        # Row 0: Original image
        axes[0, col].imshow(img_display, cmap="gray")
        axes[0, col].set_title(f"Label: {class_names[test_label]}", fontsize=9)
        axes[0, col].axis("off")

        # Row 1: Heatmap
        heatmap_up = F.interpolate(
            torch.tensor(heatmap).unsqueeze(0).unsqueeze(0).float(),
            size=img_display.shape[:2], mode="bilinear", align_corners=False
        ).squeeze().numpy()
        axes[1, col].imshow(heatmap_up, cmap="jet")
        color = "green" if pred_class == test_label else "red"
        axes[1, col].set_title(f"Pred: {class_names[pred_class]}", fontsize=9, color=color)
        axes[1, col].axis("off")

        # Row 2: Overlay
        overlay = overlay_heatmap(img_display, heatmap, alpha=0.5)
        axes[2, col].imshow(overlay)
        axes[2, col].axis("off")

    axes[0, 0].set_ylabel("Original", fontsize=11)
    axes[1, 0].set_ylabel("Grad-CAM", fontsize=11)
    axes[2, 0].set_ylabel("Overlay", fontsize=11)

    plt.suptitle("Grad-CAM Across All Digit Classes", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping (no dataset available).")

In [ ]:
# Grad-CAM for different target classes on the SAME image
if HAS_TORCHVISION:
    test_img, test_label = test_dataset[3]
    input_tensor = test_img.unsqueeze(0).to(device)
    img_display = denormalize(test_img, MEAN, STD).squeeze().numpy()

    # Generate heatmaps for different target classes
    target_classes = [0, test_label, 3, 7, 9]
    target_classes = sorted(set(target_classes))  # deduplicate

    fig, axes = plt.subplots(1, len(target_classes) + 1, figsize=(3 * (len(target_classes) + 1), 3))

    # Original
    axes[0].imshow(img_display, cmap="gray")
    axes[0].set_title(f"Original\n(Label: {class_names[test_label]})", fontsize=9)
    axes[0].axis("off")

    for i, target_cls in enumerate(target_classes):
        heatmap, _, logits = grad_cam.generate(input_tensor, target_class=target_cls)
        overlay = overlay_heatmap(img_display, heatmap, alpha=0.5)

        prob = F.softmax(logits, dim=1)[0, target_cls].item()
        axes[i + 1].imshow(overlay)
        axes[i + 1].set_title(f"Target: {class_names[target_cls]}\n(P={prob:.3f})", fontsize=9)
        axes[i + 1].axis("off")

    plt.suptitle("Grad-CAM for Different Target Classes (Same Image)", fontsize=12)
    plt.tight_layout()
    plt.show()

    print("Notice how the model focuses on different regions for different target classes.")
else:
    print("Skipping (no dataset available).")

In [ ]:
# Clean up hooks
grad_cam.remove_hooks()
print("Hooks removed.")

---

## 8. Other Explainability Methods

Grad-CAM is one of many explainability techniques. Here is a brief overview:

### Saliency Maps (Vanilla Gradients)

- Compute $\frac{\partial y^c}{\partial \mathbf{x}}$ -- the gradient of the output w.r.t. the input image
- Pixel-level importance, but can be noisy
- Simple to implement: just call `.backward()` and inspect `input.grad`

### LIME (Local Interpretable Model-agnostic Explanations)

- Perturbs input image (e.g., by occluding superpixels)
- Fits a simple linear model locally to approximate the CNN
- Model-agnostic: works with any classifier
- Library: `pip install lime`

### SHAP (SHapley Additive exPlanations)

- Based on game-theoretic Shapley values
- Assigns each feature a contribution to the prediction
- More principled but computationally expensive
- Library: `pip install shap`

### Comparison

| Method | Resolution | Speed | Model-agnostic? | Principled? |
|--------|-----------|-------|----------------|------------|
| Grad-CAM | Coarse (conv layer size) | Fast | No (needs gradients) | Moderate |
| Saliency Maps | Pixel-level | Fast | No (needs gradients) | Low (noisy) |
| LIME | Superpixel | Slow | Yes | Moderate |
| SHAP | Feature-level | Very slow | Yes | High |

In [ ]:
# Bonus: Simple saliency map implementation
if HAS_TORCHVISION:
    def compute_saliency(model, input_tensor, target_class=None):
        """Compute vanilla gradient saliency map."""
        model.eval()
        input_tensor = input_tensor.clone().requires_grad_(True)

        output = model(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()

        model.zero_grad()
        output[0, target_class].backward()

        # Take the absolute value of gradients and max across channels
        saliency = input_tensor.grad.abs().squeeze().cpu().numpy()
        if saliency.ndim == 3:
            saliency = saliency.max(axis=0)

        # Normalize
        if saliency.max() > 0:
            saliency = saliency / saliency.max()

        return saliency, target_class

    # Compare saliency map vs Grad-CAM
    test_img, test_label = test_dataset[5]
    input_tensor = test_img.unsqueeze(0).to(device)
    img_display = denormalize(test_img, MEAN, STD).squeeze().numpy()

    # Saliency map
    saliency, pred_cls = compute_saliency(model, input_tensor)

    # Grad-CAM (create new instance)
    grad_cam_temp = GradCAM(model, model.conv3)
    heatmap, _, _ = grad_cam_temp.generate(input_tensor)
    grad_cam_temp.remove_hooks()

    # Upsample Grad-CAM heatmap
    heatmap_up = F.interpolate(
        torch.tensor(heatmap).unsqueeze(0).unsqueeze(0).float(),
        size=img_display.shape[:2], mode="bilinear", align_corners=False
    ).squeeze().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(img_display, cmap="gray")
    axes[0].set_title(f"Original (Label: {class_names[test_label]})", fontsize=11)
    axes[0].axis("off")

    axes[1].imshow(saliency, cmap="hot")
    axes[1].set_title("Saliency Map (Vanilla Gradients)", fontsize=11)
    axes[1].axis("off")

    axes[2].imshow(heatmap_up, cmap="jet")
    axes[2].set_title("Grad-CAM", fontsize=11)
    axes[2].axis("off")

    plt.suptitle("Saliency Map vs Grad-CAM", fontsize=13)
    plt.tight_layout()
    plt.show()

    print("Saliency maps are pixel-level but noisier.")
    print("Grad-CAM is coarser but highlights broader relevant regions.")
else:
    print("Skipping (no dataset available).")

---

## 9. Common Mistakes and Debugging Tips

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Forgetting to call `model.eval()` | Inconsistent Grad-CAM results | Always set eval mode before generating heatmaps |
| Not zeroing gradients | Accumulated gradients from previous passes | Call `model.zero_grad()` before backward |
| Wrong target layer | Heatmap is too coarse or too fine | Use the last conv layer for best results |
| Forgetting to remove hooks | Memory leaks, slowdowns | Call `grad_cam.remove_hooks()` when done |
| Not normalizing the heatmap | Very faint or invisible heatmap | Normalize to [0, 1] before display |
| Using Grad-CAM on the input layer | Meaningless results | Grad-CAM is designed for convolutional layers |
| Not upsampling the heatmap | Tiny heatmap does not align with image | Use `F.interpolate` to match input dimensions |

---

## 10. Exercises

### Exercise 1: Apply Grad-CAM to Different Layers

Generate Grad-CAM heatmaps using `conv1`, `conv2`, and `conv3` as target layers. Compare how the heatmaps differ.

Hint: Earlier layers should produce higher-resolution but less semantically meaningful heatmaps.

In [ ]:
# ===== EXERCISE 1: Your code here =====
# target_layers = {"conv1": model.conv1, "conv2": model.conv2, "conv3": model.conv3}
# for name, layer in target_layers.items():
#     gc = GradCAM(model, layer)
#     heatmap, _, _ = gc.generate(input_tensor)
#     gc.remove_hooks()
#     # Visualize...
pass

### Exercise 2: Find Misclassified Images and Explain Them

Find 5 images that the model misclassifies. Apply Grad-CAM to understand *why* the model was confused. What regions does it focus on?

In [ ]:
# ===== EXERCISE 2: Your code here =====
# misclassified = []
# model.eval()
# for idx in range(len(test_dataset)):
#     img, label = test_dataset[idx]
#     with torch.no_grad():
#         pred = model(img.unsqueeze(0).to(device)).argmax().item()
#     if pred != label:
#         misclassified.append((idx, label, pred))
#     if len(misclassified) >= 5:
#         break
# # Apply Grad-CAM to each...
pass

### Exercise 1 -- Solution

In [ ]:
if HAS_TORCHVISION:
    test_img, test_label = test_dataset[7]
    input_tensor = test_img.unsqueeze(0).to(device)
    img_display = denormalize(test_img, MEAN, STD).squeeze().numpy()

    target_layers = {
        "conv1 (early)": model.conv1,
        "conv2 (middle)": model.conv2,
        "conv3 (late)": model.conv3,
    }

    fig, axes = plt.subplots(1, len(target_layers) + 1, figsize=(16, 4))

    axes[0].imshow(img_display, cmap="gray")
    axes[0].set_title(f"Original (Label: {class_names[test_label]})")
    axes[0].axis("off")

    for i, (name, layer) in enumerate(target_layers.items()):
        gc = GradCAM(model, layer)
        heatmap, pred_cls, _ = gc.generate(input_tensor)
        gc.remove_hooks()

        # Upsample and overlay
        overlay = overlay_heatmap(img_display, heatmap, alpha=0.5)
        axes[i + 1].imshow(overlay)
        axes[i + 1].set_title(f"{name}\nHeatmap: {heatmap.shape}", fontsize=10)
        axes[i + 1].axis("off")

    plt.suptitle("Grad-CAM at Different Layers", fontsize=13)
    plt.tight_layout()
    plt.show()

    print("Observations:")
    print("- Early layers (conv1): Higher resolution heatmap, responds to low-level features")
    print("- Later layers (conv3): Lower resolution but more semantically meaningful")
    print("- This matches the hierarchical nature of CNN feature learning")
else:
    print("Skipping (no dataset available).")

### Exercise 2 -- Solution

In [ ]:
if HAS_TORCHVISION:
    # Find misclassified images
    misclassified = []
    model.eval()

    for idx in range(len(test_dataset)):
        img, label = test_dataset[idx]
        with torch.no_grad():
            pred = model(img.unsqueeze(0).to(device)).argmax().item()
        if pred != label:
            misclassified.append((idx, label, pred))
        if len(misclassified) >= 5:
            break

    print(f"Found {len(misclassified)} misclassified images.")

    if len(misclassified) > 0:
        fig, axes = plt.subplots(2, len(misclassified), figsize=(4 * len(misclassified), 8))
        if len(misclassified) == 1:
            axes = axes.reshape(2, 1)

        gc = GradCAM(model, model.conv3)

        for col, (idx, true_label, pred_label) in enumerate(misclassified):
            img, _ = test_dataset[idx]
            input_tensor = img.unsqueeze(0).to(device)
            img_display = denormalize(img, MEAN, STD).squeeze().numpy()

            # Grad-CAM for predicted class
            heatmap, _, logits = gc.generate(input_tensor, target_class=pred_label)
            overlay = overlay_heatmap(img_display, heatmap, alpha=0.5)

            axes[0, col].imshow(img_display, cmap="gray")
            axes[0, col].set_title(
                f"True: {class_names[true_label]}\nPred: {class_names[pred_label]}",
                fontsize=10, color="red"
            )
            axes[0, col].axis("off")

            axes[1, col].imshow(overlay)
            conf = F.softmax(logits, dim=1)[0, pred_label].item()
            axes[1, col].set_title(f"Focus for '{class_names[pred_label]}'\n(conf: {conf:.2%})", fontsize=10)
            axes[1, col].axis("off")

        gc.remove_hooks()

        axes[0, 0].set_ylabel("Original", fontsize=12)
        axes[1, 0].set_ylabel("Grad-CAM", fontsize=12)
        plt.suptitle("Grad-CAM on Misclassified Images", fontsize=14)
        plt.tight_layout()
        plt.show()

        print("\nGrad-CAM helps us see WHERE the model is looking when it makes mistakes.")
        print("This can reveal if the model focuses on the wrong regions or is confused")
        print("by similar-looking features between classes.")
    else:
        print("No misclassified images found (model has perfect accuracy on tested samples).")
else:
    print("Skipping (no dataset available).")

---

**End of optional notebook.** Return to the main curriculum: [05 -- Fine-Tuning Best Practices](05_FineTuning_Best_Practices.ipynb)